In [ ]:
%pwd

'd:\\Tipto\\agentic-ai-projects\\hr-policy-agent\\notebooks'

In [2]:
import os
os.chdir("../")

In [3]:
%pwd

'd:\\Tipto\\agentic-ai-projects\\hr-policy-agent'

In [4]:
from pathlib import Path
import json, re 
from collections import Counter
import pymupdf
import pandas as pd

In [5]:
RAW = Path("data/raw")
OUT = Path("data/audit")
OUT.mkdir(exist_ok=True, parents=True)

In [6]:
# load the pdf file
PDF_PATH = RAW / "Human_Resource_Policy_Manual_PRAAN.2020.pdf"
doc = pymupdf.open(PDF_PATH)

In [7]:
type(doc)

pymupdf.Document

In [8]:
print(f"Number of pages: {doc.page_count}")
print(f"Encrypted: {doc.is_encrypted}")
print("Needs pass:", doc.needs_pass)

Number of pages: 64
Encrypted: False
Needs pass: 0


In [9]:
# get the table of contents, 3 things: level, title, page number
toc = doc.get_toc()
len(toc)

0

In [10]:
# per page features and layer verdict
OCR_FONT_HINTS = ("glyphless", "invisible", "ocr")
def page_features(page):
    text = page.get_text("text")
    rect = page.rect
    page_area = rect.width * rect.height
    
    # image converage: how much of the page the largest image occupies
    cov = []
    for info in page.get_image_info():
        r = pymupdf.Rect(info["bbox"])
        r.intersect(rect)
        cov.append(
            0 if r.is_empty else (r.width * r.height) / page_area
        )
    
    fonts = sorted({re.sub(r"^[A-Z]{6}\+", "", f[3]) for f in page.get_fonts(full=True)})
    
    # invisible text spans
    try:
        trace = page.get_texttrace()
        n_spans = len(trace)
        n_invisible = sum(1 for span in trace if span.get("type") == 3)
    except Exception:
        n_spans = -1
        n_invisible = -1
    
    return {
        "width": round(rect.width), "height": round(rect.height), "rotation": page.rotation,
        "chars": len(re.sub(r"\s+", "", text)),
        "words": len(page.get_text("words")),
        "n_blocks": len(page.get_text("blocks")),
        "n_images": len(cov), "max_img_cov": round(max(cov, default=0), 3),
        "n_spans": n_spans,
        "invisible_ratio": round(n_invisible / n_spans, 3) if n_spans > 0 else 0,
        "n_garbled": text.count("\ufffd") + len(re.findall(r"\(cid:\d+\)", text)),
        "n_private_use": sum(1 for c in text if 0xE000 <= ord(c) <= 0xF8FF),
        "n_bangla": sum(1 for c in text if 0x0980 <= ord(c) <= 0x09FF),
        "ocr_font_flag": any(h in f.lower() for f in fonts for h in OCR_FONT_HINTS),
        "fonts": "|".join(fonts),
    }

In [11]:
def classify_layer(f):
    if f["chars"] < 30 and f["n_images"] > 0:   
        return "image_only"
    if f["chars"] < 30:
        return "blank_or_vector"
    if f["invisible_ratio"] >= 0.5 or f["ocr_font_flag"]:
        return "ocr_invisible_text"
    if f["max_img_cov"] >= 0.8:
        return "scan_plus_text"
    
    return "native"

In [12]:
doc[0].get_text("text")

"oraon\nParticipatoay Research Action Nehvork- PRAAN\nEmail : pranbd.org I Phone. 07919 237'122\nwww.pranbd.org\nHuman Resource\nPolicy Manual\nApproved : December 28, 2006\nUpdated : December 28,2A20\n"

In [13]:
rows = []
for i, page in enumerate(doc, start=1):
    f = page_features(page)
    f["pdf_page"] = i
    f["label"] = page.get_label()
    rows.append(f)

In [14]:
rows

[{'width': 595,
  'height': 841,
  'rotation': 0,
  'chars': 166,
  'words': 27,
  'n_blocks': 6,
  'n_images': 1,
  'max_img_cov': 1.0,
  'n_spans': 23,
  'invisible_ratio': 1.0,
  'n_garbled': 0,
  'n_private_use': 0,
  'n_bangla': 0,
  'ocr_font_flag': False,
  'fonts': 'Helvetica',
  'pdf_page': 1,
  'label': ''},
 {'width': 595,
  'height': 841,
  'rotation': 0,
  'chars': 655,
  'words': 97,
  'n_blocks': 46,
  'n_images': 1,
  'max_img_cov': 1.0,
  'n_spans': 60,
  'invisible_ratio': 1.0,
  'n_garbled': 0,
  'n_private_use': 0,
  'n_bangla': 0,
  'ocr_font_flag': False,
  'fonts': 'Helvetica',
  'pdf_page': 2,
  'label': ''},
 {'width': 595,
  'height': 841,
  'rotation': 0,
  'chars': 921,
  'words': 181,
  'n_blocks': 68,
  'n_images': 1,
  'max_img_cov': 1.0,
  'n_spans': 121,
  'invisible_ratio': 1.0,
  'n_garbled': 0,
  'n_private_use': 0,
  'n_bangla': 0,
  'ocr_font_flag': False,
  'fonts': 'Helvetica',
  'pdf_page': 3,
  'label': ''},
 {'width': 595,
  'height': 841,
  '

In [15]:
audit = pd.DataFrame(rows).set_index("pdf_page")
audit

,width,height,rotation,chars,words,n_blocks,n_images,max_img_cov,n_spans,invisible_ratio,n_garbled,n_private_use,n_bangla,ocr_font_flag,fonts,label
pdf_page,,,,,,,,,,,,,,,,
1,595,841,0,166,27,6,1,1.0,23,1.0,0,0,0,False,Helvetica,
2,595,841,0,655,97,46,1,1.0,60,1.0,0,0,0,False,Helvetica,
3,595,841,0,921,181,68,1,1.0,121,1.0,0,0,0,False,Helvetica,
4,595,841,0,810,156,44,1,1.0,112,1.0,0,0,0,False,Helvetica,
5,595,841,0,932,179,44,1,1.0,121,1.0,0,0,0,False,Helvetica,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
60,595,841,0,2217,412,42,1,1.0,301,1.0,0,0,0,False,Helvetica,
61,595,841,0,2702,508,12,1,1.0,255,1.0,0,0,0,False,Helvetica,
62,595,841,0,825,161,8,1,1.0,97,1.0,0,0,0,False,Helvetica,


In [16]:
audit["layer_type"] = audit.apply(classify_layer, axis=1)

In [17]:
audit.to_csv(OUT / "pdf_audit.csv", index=True, encoding="utf-8")

In [18]:
print(audit["layer_type"].value_counts(), "\n")
print(audit[["chars", "words", "n_blocks", "n_images", "max_img_cov","invisible_ratio", "n_garbled", "n_private_use", "n_bangla"]].describe().T[["min", "50%", "max"]], "\n")

print("fonts:", Counter(f for s in audit["fonts"] for f in s.split("|") if f).most_common(15))
audit.head(10)

layer_type
ocr_invisible_text    64
Name: count, dtype: int64 

                  min     50%     max
chars            93.0  2171.0  2764.0
words            27.0   404.5   570.0
n_blocks          4.0    16.0    68.0
n_images          1.0     1.0     1.0
max_img_cov       1.0     1.0     1.0
invisible_ratio   1.0     1.0     1.0
n_garbled         0.0     0.0     0.0
n_private_use     0.0     0.0     0.0
n_bangla          0.0     0.0     0.0 

fonts: [('Helvetica', 64)]


,width,height,rotation,chars,words,n_blocks,n_images,max_img_cov,n_spans,invisible_ratio,n_garbled,n_private_use,n_bangla,ocr_font_flag,fonts,label,layer_type
pdf_page,,,,,,,,,,,,,,,,,
1,595,841,0,166,27,6,1,1.0,23,1.0,0,0,0,False,Helvetica,,ocr_invisible_text
2,595,841,0,655,97,46,1,1.0,60,1.0,0,0,0,False,Helvetica,,ocr_invisible_text
3,595,841,0,921,181,68,1,1.0,121,1.0,0,0,0,False,Helvetica,,ocr_invisible_text
4,595,841,0,810,156,44,1,1.0,112,1.0,0,0,0,False,Helvetica,,ocr_invisible_text
5,595,841,0,932,179,44,1,1.0,121,1.0,0,0,0,False,Helvetica,,ocr_invisible_text
6,595,841,0,1205,199,47,1,1.0,141,1.0,0,0,0,False,Helvetica,,ocr_invisible_text
7,595,841,0,794,147,42,1,1.0,107,1.0,0,0,0,False,Helvetica,,ocr_invisible_text
8,595,841,0,1818,294,21,1,1.0,162,1.0,0,0,0,False,Helvetica,,ocr_invisible_text
9,595,841,0,2509,452,13,1,1.0,237,1.0,0,0,0,False,Helvetica,,ocr_invisible_text


In [19]:
tok_re = re.compile(r"[A-Za-z]+")
page_tokens = {p: tok_re.findall(pg.get_text("text")) for p, pg in enumerate(doc, start=1)}
vocab = Counter(t.lower() for toks in page_tokens.values() for t in toks)

rows = []
for p, toks in page_tokens.items():
    for a, b in zip(toks, toks[1:]):
        joined = (a + b).lower()
        if len(joined) >= 5 and vocab[joined] >= 2 and (vocab[a.lower()] <= 2 or vocab[b.lower()] <= 2):
            rows.append({"pdf_page": p, "split": f"{a} {b}", "likely": joined, "likely_freq": vocab[joined]})

anom = pd.DataFrame(rows)
anom.to_csv(OUT / "split_word_candidates.csv", index=False, encoding="utf-8")
print(len(anom), "candidates on", anom["pdf_page"].nunique() if len(anom) else 0, "pages")
anom.head(30)

18 candidates on 14 pages


,pdf_page,split,likely,likely_freq
0,8,pran bd,pranbd,4
1,9,polic es,polices,2
2,14,off icer,officer,16
3,21,tion s,tions,2
4,22,up dated,updated,4
5,31,grad e,grade,31
6,33,Tota l,total,3
7,37,su pervisor,supervisor,26
8,37,hand over,handover,2
9,46,su pervisor,supervisor,26


In [20]:
print("page_count:", doc.page_count, "| audit rows:", len(audit))

print(audit["layer_type"].value_counts(), "\n")
print("pages NOT ocr_invisible_text:", audit.index[audit["layer_type"] != "ocr_invisible_text"].tolist())
print("low-text pages (<300 chars):", audit.index[audit["chars"] < 300].tolist(), "\n")
print(audit["chars"].describe(), "\n")
print("metadata:", json.dumps(doc.metadata, indent=2, ensure_ascii=False))
print("outline entries:", len(doc.get_toc()))

info = doc[0].get_image_info(xrefs=True)[0]
print({k: info[k] for k in ("width", "height", "xres", "yres", "bpc", "colorspace")})

page_count: 64 | audit rows: 64
layer_type
ocr_invisible_text    64
Name: count, dtype: int64 

pages NOT ocr_invisible_text: []
low-text pages (<300 chars): [1, 26, 32] 

count      64.000000
mean     1855.515625
std       727.337392
min        93.000000
25%      1283.500000
50%      2171.000000
75%      2415.750000
max      2764.000000
Name: chars, dtype: float64 

metadata: {
  "format": "PDF 1.3",
  "title": "",
  "author": "",
  "subject": "",
  "keywords": "",
  "creator": "Canon SC1011",
  "producer": "IJ Scan Utility",
  "creationDate": "D:20210717170751+06'00'",
  "modDate": "",
  "trapped": "",
  "encryption": null
}
outline entries: 0
{'width': 1240, 'height': 1753, 'xres': 96, 'yres': 96, 'bpc': 8, 'colorspace': 3}


In [21]:
targets = {
    "s11_disciplinary": "Disciplinary Manners",
    "s10_travel_table": "Accommodation and",
    "pf_refund_table": "Refund/payment of PF",
}
found = {k: [p for p, pg in enumerate(doc, 1) if v.lower() in pg.get_text().lower()]
         for k, v in targets.items()}
print(found)

IMG = OUT / "pages"; IMG.mkdir(exist_ok=True)
for pages in found.values():
    for p in pages[:2]:
        doc[p - 1].get_pixmap(dpi=200).save(IMG / f"page_{p:03d}.png")

# What does the layer look like on the per diem table page?
p = found["s10_travel_table"][0]
words = doc[p - 1].get_text("words") # (x0, y0, x1, y1, word, block, line, word_no)
print("page", p, "| words:", len(words), "| first 12:", [(w[4], round(w[0]), round(w[1])) for w in words[:12]])
print(doc[p - 1].get_text("text")[:2500])

{'s11_disciplinary': [5, 44], 's10_travel_table': [5, 41], 'pf_refund_table': []}
page 5 | words: 179 | first 12: [('Section', 92, 179), ('09', 133, 179), ('Section', 93, 360), ("'10", 134, 362), ('Section', 93, 486), ('l1', 135, 487), ("B.'1", 159, 100), ('Policy', 197, 106), ('Statement', 228, 106), ('B-2', 159, 118), ('Position,', 197, 125), ('Grade', 240, 124)]
Section 09
Section '10
Section l1
B.'1 Policy Statement
B-2 Position, Grade and Salary Structure
8.3 
Review oi Salary Structlre
a.4 
Remuneration and Benefit Package
Leave Policy
9.1 
General
9-2 
General Leave Policies
9.3 
Casual Leave (Core and Project Staf0
9.4 
Earned leave
C 5 
Sick Leave
9.6 
lvlaternity Leave
9.7 
Paternity Leave
9.8 
Leave without Pay
9.9 
Procedure for Applying for and Availing of Leave
Staft Travel, Accommodation and Perdiem
10.1 Policy Statement
10.2 Travel, Accommodation and Perdrem
10.3 Phone Calls/l\,4obile Phone Use
10.4 Pocket Alowance
10.5 Travel Procedure
10.6 Travel & Per Diem Advance an

In [22]:
IMG = OUT / "pages"
IMG.mkdir(exist_ok = True)

In [24]:
sizes = set()
for p in range(1, doc.page_count + 1):
    xref = doc[p - 1].get_images(full=True)[0][0]
    d = doc.extract_image(xref)
    
    (IMG / f"page_{p:03d}.{d['ext']}").write_bytes(d["image"])
    sizes.add((d["width"], d["height"], d["ext"], d["colorspace"]))
    
print("distinct (w, h, format, channels):", sizes)
print("total MB:", round(sum(f.stat().st_size for f in IMG.glob('page_*')) / 1e6, 1))

distinct (w, h, format, channels): {(1240, 1753, 'jpeg', 3)}
total MB: 21.7


In [25]:
import cv2
import numpy as np

In [30]:
def read_bgr(p):
    f = next(IMG.glob(f"page_{p:03d}.*"))
    return cv2.imdecode(np.fromfile(str(f), dtype=np.uint8), cv2.IMREAD_COLOR)

In [31]:
def skew_angle(ink):
    small = cv2.resize(ink, None, fx=0.5, fy=0.5, interpolation=cv2.INTER_AREA)
    h, w = small.shape
    best , best_score = 0.0, -1.0
    
    for a in np.arange(-3, 3.01, 0.25):
        M = cv2.getRotationMatrix2D((w/2, h/2), a, 1)
        r = cv2.warpAffine(small, M, (w, h), flags=cv2.INTER_NEAREST)
        
        score = np.var(r.sum(axis=1))
        if score > best_score:
            best_score = score
            best = float(a)
    
    return best # rotation that best straightens the text lines

In [32]:
def image_features(bgr):
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    _, ink = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    h, w = ink.shape
    hl = cv2.morphologyEx(ink, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_RECT, (w // 8, 1)))
    vl = cv2.morphologyEx(ink, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_RECT, (1, h // 15)))
    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
    blue = cv2.inRange(hsv, (100, 80, 50), (135, 255, 255))
    return {
        "ink_ratio": round(float((ink > 0).mean()), 4),
        "h_lines": int(cv2.connectedComponents(hl)[0] - 1),
        "v_lines": int(cv2.connectedComponents(vl)[0] - 1),
        "blue_ratio": round(float((blue > 0).mean()), 5),
        "brightness": round(float(gray.mean()), 1),
        "contrast": round(float(gray.std()), 1),
        "skew_deg": skew_angle(ink),
    }

In [33]:
imgf = pd.DataFrame([
    {"pdf_page": p, **image_features(read_bgr(p))}
    for p in range(1, doc.page_count + 1)]).set_index("pdf_page")

In [34]:
feat = audit.join(imgf)

In [35]:
feat["chars_per_ink"] = (feat["chars"] / (feat["ink_ratio"] * 1000)).round(2)

In [36]:
feat

,width,height,rotation,chars,words,n_blocks,n_images,max_img_cov,n_spans,invisible_ratio,...,label,layer_type,ink_ratio,h_lines,v_lines,blue_ratio,brightness,contrast,skew_deg,chars_per_ink
pdf_page,,,,,,,,,,,,,,,,,,,,,
1,595,841,0,166,27,6,1,1.0,23,1.0,...,,ocr_invisible_text,0.3472,6,1,0.19326,184.2,95.2,0.00,0.48
2,595,841,0,655,97,46,1,1.0,60,1.0,...,,ocr_invisible_text,0.0206,3,0,0.00172,249.2,33.8,0.00,31.80
3,595,841,0,921,181,68,1,1.0,121,1.0,...,,ocr_invisible_text,0.0280,4,0,0.00153,247.1,39.4,-0.25,32.89
4,595,841,0,810,156,44,1,1.0,112,1.0,...,,ocr_invisible_text,0.0249,2,0,0.00163,248.0,37.1,0.00,32.53
5,595,841,0,932,179,44,1,1.0,121,1.0,...,,ocr_invisible_text,0.0282,4,0,0.00165,247.1,39.4,-0.25,33.05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
60,595,841,0,2217,412,42,1,1.0,301,1.0,...,,ocr_invisible_text,0.0728,0,0,0.00165,234.3,61.9,-2.25,30.45
61,595,841,0,2702,508,12,1,1.0,255,1.0,...,,ocr_invisible_text,0.0717,3,0,0.00186,234.8,61.6,-0.25,37.68
62,595,841,0,825,161,8,1,1.0,97,1.0,...,,ocr_invisible_text,0.0241,3,0,0.00176,248.2,36.7,-0.25,34.23


In [37]:
feat.to_csv(
    OUT/"page_features.csv",
    encoding="utf-8"
)

In [39]:
TEST_PAGES = [5, 16, 32, 34, 41, 42, 44, 56]
ROTATE_CW = {32}                       
PREP = OUT / "pages_prep"; PREP.mkdir(exist_ok=True)

In [40]:
def deskew(bgr):
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    _, ink = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    a = skew_angle(ink)
    h, w = bgr.shape[:2]
    M = cv2.getRotationMatrix2D((w / 2, h / 2), a, 1.0)
    return cv2.warpAffine(bgr, M, (w, h), flags=cv2.INTER_CUBIC, borderValue=(255, 255, 255)), a

In [41]:
def suppress_blue(bgr):
    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
    mask = cv2.dilate(cv2.inRange(hsv, (100, 80, 50), (135, 255, 255)), np.ones((3, 3), np.uint8))
    out = bgr.copy(); out[mask > 0] = (255, 255, 255)
    return out

In [42]:
angles = {}
for p in TEST_PAGES:
    bgr = read_bgr(p)
    if p in ROTATE_CW:
        bgr = cv2.rotate(bgr, cv2.ROTATE_90_CLOCKWISE)
    
    bgr, angles[p] = deskew(bgr)
    bgr = suppress_blue(bgr)
    cv2.imencode(".png", bgr)[1].tofile(str(PREP / f"page_{p:03d}.png"))

print("deskew angles applied:", angles)

deskew angles applied: {5: -0.25, 16: -1.0, 32: -0.25, 34: -0.25, 41: -0.25, 42: -0.25, 44: -0.25, 56: 0.0}


In [43]:
# run RapidOCR + GLM-OCR through ollama
import time 
import ollama

OCR = OUT / "ocr"
for sub in ("rapid_raw", "rapid_prep", "glm_prep"):
    (OCR / sub).mkdir(exist_ok=True, parents=True)

In [44]:
from rapidocr_onnxruntime import RapidOCR
rapid = RapidOCR()

In [45]:
def read_img(path):
    return cv2.imdecode(
        np.fromfile(str(path), dtype = np.uint8), 
        cv2.IMREAD_COLOR
    )

In [46]:
def run_rapid(bgr):
    result, _ = rapid(bgr)
    return [] if result is None else [
        {"box": [[float(x), float(y)] for x, y in b], "text": t, "score": float(s)} for b, t, s in result]

In [47]:
def items_to_text(items, y_tol=10):
    ys = lambda it: float(np.mean([pt[1] for pt in it["box"]]))
    lines, cur, cur_y = [], [], None
    for it in sorted(items, key=lambda i: (ys(i), i["box"][0][0])):
        if cur_y is None or abs(ys(it) - cur_y) <= y_tol:
            cur.append(it); cur_y = ys(it) if cur_y is None else (cur_y + ys(it)) / 2
        else:
            lines.append(cur); cur, cur_y = [it], ys(it)
    if cur: lines.append(cur)
    return "\n".join(" ".join(i["text"] for i in sorted(l, key=lambda i: i["box"][0][0])) for l in lines)

In [55]:
def split_strips(bgr, max_h=550):
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    _, ink = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    frac = (ink > 0).mean(axis=1) # ink fraction per pixel row
    ok = (frac < 0.004) | (frac > 0.6) # blank rows or full-width ruled lines
    h, cuts, start = len(frac), [], 0
    while h - start > max_h:
        window = np.arange(start + max_h // 2, min(start + max_h, h))
        cand = window[ok[window]]
        cut = int(cand[-1]) if len(cand) else start + max_h   # last clean cut inside the window
        cuts.append(cut); start = cut
    edges = [0] + cuts + [h]
    return [bgr[a:b] for a, b in zip(edges[:-1], edges[1:]) if b - a > 20]

In [56]:
def glm_call(image, opts):
    r = ollama.chat(model="glm-ocr",
                    messages=[{"role": "user", "content": "Text Recognition:", "images": [image]}],
                    options={"num_ctx": 8192, "num_predict": 4096, **opts})
    return r["message"]["content"]

In [57]:

def run_glm(prep_path):
    try: # 1) whole page, greedy
        return glm_call(str(prep_path), {"temperature": 0}), "page"
    except ollama.ResponseError:
        pass
    parts = [] # 2) fall back to strips
    for strip in split_strips(read_img(prep_path)):
        png = cv2.imencode(".png", strip)[1].tobytes()
        try:
            parts.append(glm_call(png, {"temperature": 0}))
        except ollama.ResponseError:
            try:
                parts.append(glm_call(png, {"temperature": 0.2, "repeat_penalty": 1.05, "top_p": 0.9}))
            except ollama.ResponseError as e:
                parts.append(f"[STRIP FAILED: {str(e)[:60]}]")
    return "\n".join(parts), "strips"

In [58]:
timing, modes = [], {}
for p in TEST_PAGES:
    prep_path = PREP / f"page_{p:03d}.png"
    for tag, load in (("rapid_raw", lambda: read_bgr(p)), ("rapid_prep", lambda: read_img(prep_path))):
        f = OCR / tag / f"page_{p:03d}.txt"
        if f.exists():
            continue
        t0 = time.time(); items = run_rapid(load()); dt = time.time() - t0
        (OCR / tag / f"page_{p:03d}.json").write_text(json.dumps(items, ensure_ascii=False), encoding="utf-8")
        f.write_text(items_to_text(items), encoding="utf-8")
        timing.append((p, tag, round(dt, 1)))
    f = OCR / "glm_prep" / f"page_{p:03d}.txt"
    if not f.exists():
        t0 = time.time(); txt, modes[p] = run_glm(prep_path); dt = time.time() - t0
        f.write_text(txt, encoding="utf-8")
        timing.append((p, "glm_prep", round(dt, 1)))
    print("page", p, "done", modes.get(p, "(cached)"))

print("\nGLM mode per page:", modes)
print("failed strips:", sum("[STRIP FAILED" in (OCR / "glm_prep" / f"page_{p:03d}.txt").read_text(encoding="utf-8") for p in TEST_PAGES))

page 5 done strips
page 16 done strips
page 32 done strips
page 34 done strips
page 41 done strips
page 42 done page
page 44 done strips
page 56 done strips

GLM mode per page: {5: 'strips', 16: 'strips', 32: 'strips', 34: 'strips', 41: 'strips', 42: 'page', 44: 'strips', 56: 'strips'}
failed strips: 1


In [59]:
def load_txt(tag, p): 
    return (OCR / tag / f"page_{p:03d}.txt").read_text(encoding="utf-8")


num_re = re.compile(r"\d(?:[\d,\.]*\d)?")
def numbers(text): 
    return [t.replace(",", "") for t in num_re.findall(text)]

def bag(text): 
    return Counter(numbers(text))

def agree(a, b): 
    return round(2 * sum((a & b).values()) / max(1, sum(a.values()) + sum(b.values())), 3)

In [60]:
def show_diff(p, a="rapid_prep", b="glm_prep", k=15):
    A, B = bag(load_txt(a, p)), bag(load_txt(b, p))
    print(f"page {p} | only in {a}: {list((A - B).elements())[:k]}")
    print(f"page {p} | only in {b}: {list((B - A).elements())[:k]}\n")

In [61]:
for p in TEST_PAGES:
    t = load_txt("glm_prep", p)
    if "[STRIP FAILED" in t:
        print("failed strip on page", p, "->", [l for l in t.splitlines() if "[STRIP FAILED" in l])

print("\n--- GLM, page 44 (first 14 lines) ---")
print("\n".join(load_txt("glm_prep", 44).splitlines()[:14]))
print("\n--- RapidOCR (prepped), page 44 (first 14 lines) ---")
print("\n".join(load_txt("rapid_prep", 44).splitlines()[:14]))

failed strip on page 32 -> ['[STRIP FAILED: prediction aborted, token repeat limit reached (status code:]', '[STRIP FAILED: prediction aborted, token repeat limit reached (status code:]']

--- GLM, page 44 (first 14 lines) ---
Section 11: Disciplinary Manners

11.1: Disciplinary Action

a. By accepting an appointment with PRAAN all employees agree to work in a responsible, disciplined, harmonious and productive manner, to be loyal to the organization and to act in a manner conducive to the accomplishment of the organizational objectives.

b. The main objective of disciplining is to punish the wrong doers and help employees to improve their behavior that led to improved performances and not to alienate him/her from the organization.

c. The objective of disciplinary action should be to motivate the employee to the extent possible to improve performance. Disciplinary actions are not taken just
Section 11: Disciplinary Manners

11.1: Disciplinary Action

a. By accepting an appointment wit

In [62]:
rows = []
for p in TEST_PAGES:
    texts = {"embedded": doc[p - 1].get_text(), "rapid_raw": load_txt("rapid_raw", p),
             "rapid_prep": load_txt("rapid_prep", p), "glm_prep": load_txt("glm_prep", p)}
    b = {k: bag(v) for k, v in texts.items()}
    rows.append({"page": p, **{f"n_{k}": sum(v.values()) for k, v in b.items()},
                 "rapid_prep~glm": agree(b["rapid_prep"], b["glm_prep"]),
                 "rapid_raw~prep": agree(b["rapid_raw"], b["rapid_prep"]),
                 "embedded~glm": agree(b["embedded"], b["glm_prep"])})
display(pd.DataFrame(rows).set_index("page"))

def show_diff(p, a="rapid_prep", b="glm_prep", k=15):
    A, B = bag(load_txt(a, p)), bag(load_txt(b, p))
    print(f"page {p} | only in {a}: {list((A - B).elements())[:k]}")
    print(f"page {p} | only in {b}: {list((B - A).elements())[:k]}\n")
for p in (5, 41, 44):
    show_diff(p)

,n_embedded,n_rapid_raw,n_rapid_prep,n_glm_prep,rapid_prep~glm,rapid_raw~prep,embedded~glm
page,,,,,,,
5,74,81,81,1859,0.084,1.000,0.062
16,21,19,21,573,0.064,0.850,0.040
32,8,370,410,810,0.000,0.669,0.002
34,11,16,17,68,0.376,0.970,0.253
41,49,44,41,362,0.199,0.941,0.195
42,53,55,56,839,0.049,0.973,0.036
44,46,33,32,754,0.079,0.954,0.052
56,29,22,21,473,0.081,0.977,0.104


page 5 | only in rapid_prep: []
page 5 | only in glm_prep: ['8.1', '8.1', '8.1', '8.1', '8.1', '8.1', '8.1', '8.1', '8.1', '8.1', '8.1', '8.1', '8.1', '8.1', '8.1']

page 41 | only in rapid_prep: ['105']
page 41 | only in glm_prep: ['10', '10', '10', '10', '10', '10', '10', '10', '10', '10', '10', '10', '10', '10', '10']

page 44 | only in rapid_prep: ['3']
page 44 | only in glm_prep: ['11', '11', '11', '11', '11', '11', '11', '11', '11', '11', '11', '11', '11', '11', '11']



In [63]:
basic = dict(A=55000, B=45000, C=35000, D=30000, E=25000, F=18000, G=12000, H=10000, I=5000)
gross = dict(A=75000, B=65000, C=50000, D=40000, E=35000, F=30000, G=22000, H=15000, I=10000)
cells = [(f"{t}-{g}", n + 1, {int(v * 1.05 ** n), int(v * 1.05 ** n) + 1})
         for t, d in (("basic", basic), ("gross", gross)) for g, v in d.items() for n in range(20)]

def cell_recall(text):
    pool = Counter(int(t) for t in numbers(text) if t.isdigit() and len(t) >= 4)
    hit = 0
    for _, _, cands in cells:
        for c in cands:
            if pool[c] > 0:
                pool[c] -= 1; hit += 1; break
    return hit

for tag, text in {"embedded": doc[31].get_text(), "rapid_raw": load_txt("rapid_raw", 32),
                  "rapid_prep": load_txt("rapid_prep", 32), "glm_prep": load_txt("glm_prep", 32)}.items():
    print(f"{tag:11s} {cell_recall(text):3d} / {len(cells)} salary cells matched")

embedded      0 / 360 salary cells matched
rapid_raw   166 / 360 salary cells matched
rapid_prep  110 / 360 salary cells matched
glm_prep      0 / 360 salary cells matched


In [64]:
print("deskew angles applied:", angles)

# (a) page 32: what did RapidOCR read, and what did it miss?
t32 = load_txt("rapid_prep", 32)
print("\n".join(t32.splitlines()[:10]), "\n")
pool = Counter(numbers(t32))
missing = [(nm, n, min(c)) for nm, n, c in cells if not any(pool[str(v)] for v in c)]
odd = [t for t in pool if not t.isdigit()]
print(len(missing), "cells missing | sample:", missing[:12])
print(len(odd), "non-integer tokens | sample:", odd[:20], "\n")

# (b) missing-spaces metric: letter-only tokens longer than 20 characters
def long_tokens(text): return sum(len(t) > 20 for t in re.findall(r"[A-Za-z]+", text))
print("long tokens (raw, prep, glm):", {p: (long_tokens(load_txt("rapid_raw", p)),
      long_tokens(load_txt("rapid_prep", p)), long_tokens(load_txt("glm_prep", p))) for p in TEST_PAGES}, "\n")

# (c) GLM repetition: total vs unique lines
for p in TEST_PAGES:
    lines = [l.strip() for l in load_txt("glm_prep", p).splitlines() if l.strip()]
    print("page", p, "| glm lines:", len(lines), "| unique:", len(set(lines)))

# (d) page 41: how does each engine read the grade A row (the truncated total)?
for tag in ("rapid_raw", "rapid_prep", "glm_prep"):
    print(tag, [l[:120] for l in load_txt(tag, 41).splitlines() if "300" in l][:3])

deskew angles applied: {5: -0.25, 16: -1.0, 32: -0.25, 34: -0.25, 41: -0.25, 42: -0.25, 44: -0.25, 56: 0.0}
SalaryStructurewithGrade&StepsofPRAAN:
AllStaffsofPRAANareentitled toreceivetheirSalarybyfollowingGradesandSteps.ManagementofPRAANwilidecidedtr:at theGradeandStepconsideringtheir
nci nbo Licip yearsofexperienceandrequirementsofthejob.TheSalarystructureofPRAANisgivenbellow:
Po Salary Structureof PRAAN(BasicSalary)
Grade Steps
Exe Ma com 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20
cuti ual
Cor 2020 A 55,000 57.750 60.638 63.669 66.853 70.195 73.705 77.391 81.260 85.323 89.589 94,069 98.772 103.711 108,896 114.341 120,058 126,061 132,364 138.982
nmittee pranbd. Netwo B 45.000 47,250 49.613 52,093 54.398 57,433 60.304 63.320 56.485 69,810 73.300 76.965 80.814 84,854 89,097 93.552 98.229 103.141 108.298 113.713
ork- 

221 cells missing | sample: [('basic-A', 2, 57750), ('basic-A', 3, 60637), ('basic-A', 4, 63669), ('basic-A', 5, 66852), ('basic-A', 6, 70195), ('basic-A', 7, 737

In [65]:
strip = split_strips(read_img(PREP / "page_044.png"))[0]
png = cv2.imencode(".png", strip)[1].tobytes()
variants = {
    "greedy_cap800": {"temperature": 0, "num_predict": 800},
    "greedy_rep1.1": {"temperature": 0, "num_predict": 800, "repeat_penalty": 1.1},
    "t0.1_rep1.05":  {"temperature": 0.1, "top_p": 0.9, "num_predict": 800, "repeat_penalty": 1.05},
}
for name, o in variants.items():
    t0 = time.time()
    try:
        r = ollama.chat(model="glm-ocr",
                        messages=[{"role": "user", "content": "Text Recognition:", "images": [png]}],
                        options={"num_ctx": 8192, **o})
        txt = r["message"]["content"]
        print(f"{name:15s} done_reason={r.get('done_reason')} tokens={r.get('eval_count')} "
              f"chars={len(txt)} sec={time.time() - t0:.1f}")
        print("    ends with:", repr(txt[-70:]))
    except ollama.ResponseError as e:
        print(f"{name:15s} ERROR {str(e)[:70]}")

greedy_cap800   done_reason=length tokens=800 chars=3302 sec=5.6
    ends with: 'objective of disciplinary action should be to motivate the employee to'
greedy_rep1.1   done_reason=length tokens=800 chars=3334 sec=3.3
    ends with: 'n should be to motivate the employee to the extent possible to improve'
t0.1_rep1.05    done_reason=length tokens=800 chars=3302 sec=3.2
    ends with: 'objective of disciplinary action should be to motivate the employee to'


In [66]:
from rapidocr import RapidOCR, LangRec, ModelType, OCRVersion

def to_items(res):
    if res is None or res.txts is None:
        return []
    return [{"box": [[float(x), float(y)] for x, y in b], "text": t, "score": float(s)}
            for b, t, s in zip(res.boxes, res.txts, res.scores)]

for ver in (OCRVersion.PPOCRV4, OCRVersion.PPOCRV5):
    for mt in (ModelType.MOBILE, ModelType.SERVER):
        try:
            eng = RapidOCR(params={"Rec.lang_type": LangRec.EN, "Rec.model_type": mt, "Rec.ocr_version": ver})
            txt = items_to_text(to_items(eng(read_bgr(44))))          # raw page 44, no preprocessing
            print(ver.name, mt.name, "| long tokens:", long_tokens(txt), "| numbers:", len(numbers(txt)), "| chars:", len(txt))
            print("    ", txt.splitlines()[:2])
        except Exception as e:
            print(ver.name, mt.name, "unavailable:", str(e)[:90])

2026-09-20 09:46:31,607 - RapidOCR - INFO: Using engine_name: onnxruntime
2026-09-20 09:46:31,621 - RapidOCR - INFO: File exists and is valid: D:\Tipto\agentic-ai-projects\hr-policy-agent\.venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
2026-09-20 09:46:31,623 - RapidOCR - INFO: Using D:\Tipto\agentic-ai-projects\hr-policy-agent\.venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
2026-09-20 09:46:31,683 - RapidOCR - INFO: Using engine_name: onnxruntime
2026-09-20 09:46:31,683 - RapidOCR - INFO: File exists and is valid: D:\Tipto\agentic-ai-projects\hr-policy-agent\.venv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
2026-09-20 09:46:31,683 - RapidOCR - INFO: Using D:\Tipto\agentic-ai-projects\hr-policy-agent\.venv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
2026-09-20 09:46:31,740 - RapidOCR - INFO: Using engine_name: onnxruntime
2026-09-20 09:46:31,741 - RapidOCR - INFO: Initiating download: https://www.mode

PPOCRV4 MOBILE | long tokens: 0 | numbers: 25 | chars: 2468
     ['Section 11:Disciplinary Manners', '11.1Disciplinary Action']


2026-09-20 09:46:43,389 - RapidOCR - INFO: Using engine_name: onnxruntime
2026-09-20 09:46:43,389 - RapidOCR - ERROR: Unsupported configuration:
  engine_type   = onnxruntime
  ocr_version   = PP-OCRv4
  task_type     = rec
  lang_type     = en
  model_type     = server

Please refer to the official model list for supported combinations:
https://rapidai.github.io/RapidOCRDocs/main/model_list/

Example valid usage:
  from rapidocr import LangRec, OCRVersion, RapidOCR
  engine = RapidOCR(params={'Rec.ocr_version': OCRVersion.PPOCRV5, 'Rec.lang_type': LangRec.CH, 'Rec.model_type': 'mobile'})
2026-09-20 09:46:43,424 - RapidOCR - INFO: Using engine_name: onnxruntime
2026-09-20 09:46:43,434 - RapidOCR - INFO: File exists and is valid: D:\Tipto\agentic-ai-projects\hr-policy-agent\.venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
2026-09-20 09:46:43,435 - RapidOCR - INFO: Using D:\Tipto\agentic-ai-projects\hr-policy-agent\.venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small

PPOCRV4 SERVER unavailable: Invalid OCR configuration.


2026-09-20 09:46:43,584 - RapidOCR - INFO: Initiating download: https://www.modelscope.cn/models/RapidAI/RapidOCR/resolve/v3.9.2/onnx/PP-OCRv5/rec/en_PP-OCRv5_rec_mobile.onnx
2026-09-20 09:46:45,297 - RapidOCR - INFO: Download size: 7.51MB
2026-09-20 09:46:46,231 - RapidOCR - INFO: Successfully saved to: D:\Tipto\agentic-ai-projects\hr-policy-agent\.venv\Lib\site-packages\rapidocr\models\en_PP-OCRv5_rec_mobile.onnx
2026-09-20 09:46:46,233 - RapidOCR - INFO: Using D:\Tipto\agentic-ai-projects\hr-policy-agent\.venv\Lib\site-packages\rapidocr\models\en_PP-OCRv5_rec_mobile.onnx
2026-09-20 09:46:50,595 - RapidOCR - INFO: Using engine_name: onnxruntime
2026-09-20 09:46:50,602 - RapidOCR - INFO: File exists and is valid: D:\Tipto\agentic-ai-projects\hr-policy-agent\.venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
2026-09-20 09:46:50,602 - RapidOCR - INFO: Using D:\Tipto\agentic-ai-projects\hr-policy-agent\.venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
2026-09-2

PPOCRV5 MOBILE | long tokens: 0 | numbers: 25 | chars: 2513
     ['Section 11 : Disciplinary Manners', '11.1:Disciplinary Action']
PPOCRV5 SERVER unavailable: Invalid OCR configuration.
